# Table of Contents

- [Section 1: Concepts and Practical Applications](#Section-1:-Concepts-and-Practical-Applications)
  - [1. What Is CUPED?](#1.-What-Is-CUPED?)
  - [2. How CUPED Works](#2.-How-CUPED-Works)
  - [3. Why CUPED Matters](#3.-Why-CUPED-Matters)
    - [3.1 Why CUPED When Large Tech Companies Already Have Millions of Users?](#3.1-Why-CUPED-When-Large-Tech-Companies-Already-Have-Millions-of-Users?)
    - [3.2 When Is CUPED Particularly Valuable?](#3.2-When-Is-CUPED-Particularly-Valuable?)
    - [3.3 When Is CUPED Less Necessary?](#3.3-When-Is-CUPED-Less-Necessary?)
  - [4. How to use CUPED in Causal Inference?](#4.-How-to-use-CUPED-in-Causal-Inference?)
    - [4.1 Randomized A/B Experiments: The Primary Use Case](#4.1-Randomized-A/B-Experiments:-The-Primary-Use-Case)
    - [4.2 Observational Studies: Not a Standalone Causal Inference Method](#4.2-Observational-Studies:-Not-a-Standalone-Causal-Inference-Method)
  - [5. Similar Variance Reduction Methods](#5.-Similar-Variance-Reduction-Methods)
    - [5.1 Rank by Industry Usage Frequency](#5.1-Rank-by-Industry-Usage-Frequency)
    - [5.2 CUPED / ANCOVA: The Standard Starting Point](#5.2-CUPED-/-ANCOVA:-The-Standard-Starting-Point)
    - [5.3 Multivariate CUPED / Regression Adjustment](#5.3-Multivariate-CUPED-/-Regression-Adjustment)
    - [5.4 CUPAC: Machine Learning-Based Variance Reduction](#5.4-CUPAC:-Machine-Learning-Based-Variance-Reduction)
  - [6. Practical Decision Guide](#6.-Practical-Decision-Guide)
- [Section 2. CUPED: Simulation and Implementation](#Section-2.-CUPED:-Simulation-and-Implementation)
  - [2.1 Objective](#2.1-Objective)
  - [2.2 Simulation Setup](#2.2-Simulation-Setup)
  - [2.3 Results and Interpretation](#2.3-Results-and-Interpretation)
    - [Treatment Effect Estimation](#Treatment-Effect-Estimation)
    - [Variance Reduction](#Variance-Reduction)
    - [Confidence Interval Comparison](#Confidence-Interval-Comparison)
    - [Key Takeaway](#Key-Takeaway)

# Section 1: Concepts and Practical Applications

# 1. What Is CUPED?

**CUPED (Controlled Experiments Using Pre-Experiment Data)** is a variance reduction technique introduced by Microsoft researchers in 2013 to improve the statistical sensitivity of randomized A/B experiments.

The main idea is to **Use information about users before an experiment to explain predictable variation in their outcomes during the experiment.**

For example, suppose a ride-sharing App runs an A/B test to determine whether a new rider feature increases the number of completed trips. Even without the new feature, different users naturally have different levels of activity:

- Some riders complete 30 trips per month.
- Others complete only 2 trips per month.
- Some users have been active for years, while others are relatively new.

This natural variation creates noise in the experiment's outcome metric. However, historical behavior can help explain some of this variation: if a rider completed 25 trips during the previous 28 days, we might reasonably expect them to complete more trips during the experiment than someone who previously completed only 2 trips. CUPED uses this historical information to adjust the post-experiment outcome, reducing noise without removing the treatment effect.

# 2. How CUPED Works

Let:

- $Y$ = Post-experiment outcome, such as trips completed during the experiment.
- $X$ = Pre-experiment covariate, such as trips completed during the previous 28 days.
- $\theta$ = Adjustment coefficient.

The CUPED-adjusted outcome is:

$$
Y_{\text{CUPED}} = Y - \theta(X - E[X])
$$

where:

$$
\theta = \frac{\operatorname{Cov}(Y,X)}{\operatorname{Var}(X)}
$$

The adjustment removes the portion of outcome variation that can be explained by the historical covariate.

For a single optimally adjusted covariate, the theoretical variance reduction is approximately:

$$
\text{Variance Reduction} = \rho_{X,Y}^2
$$

where $\rho_{X,Y}$ is the correlation between the historical covariate and the post-experiment outcome.

For example:

| Correlation Between X and Y | Theoretical Variance Reduction |
|---|---|
| 0.9 | 81% |
| 0.7 | 49% |
| 0.5 | 25% |
| 0.2 | 4% |

These are theoretical results under the standard linear control-variate setup, not guaranteed improvements for every experiment.

# 3. Why CUPED Matters

By reducing outcome variance, CUPED can:

1. **Reduce standard errors**, resulting in narrower confidence intervals.
2. **Increase statistical power**, making smaller treatment effects easier to detect.
3. **Reduce required sample size**, allowing some experiments to reach a target level of precision with fewer users.
4. **Potentially shorten experiment duration**, provided the experiment still satisfies its minimum runtime and other design requirements.

CUPED does not create a more comparable Treatment and Control group. Randomization already provides that foundation. CUPED improves how precisely we estimate the treatment effect by reducing predictable outcome variation.

## 3.1 Why CUPED When Large Tech Companies Already Have Millions of Users?

Large technology companies often have access to millions of users, so it is reasonable to ask: if we already have a sufficiently large sample size, why do we need CUPED?

The primary benefit of CUPED is not simply reducing the number of users required for an experiment. **Its main benefit is improving the precision of treatment effect estimates by reducing standard errors and narrowing confidence intervals.**

For a randomized experiment, the approximate 95% confidence interval is:

$$
\hat{\tau} \pm 1.96 \times SE(\hat{\tau})
$$

where:

- $\hat{\tau}$ is the estimated treatment effect.
- $SE(\hat{\tau})$ is the standard error of the treatment effect estimate.

CUPED reduces outcome variance, which can reduce the standard error of the estimated treatment effect. As a result, the confidence interval becomes narrower, and the precision improves.

For example:

| Method | Estimated Treatment Effect | 95% Confidence Interval |
|---|---|---|
| Without CUPED | +$0.50 | [-$0.10, $1.10] |
| With CUPED | +$0.50 | [$0.10, $0.90] |

In this example, CUPED improves precision enough for the confidence interval to exclude zero. However, CUPED does not guarantee statistical significance, and the adjusted point estimate may also differ from the unadjusted estimate.

##3.2 When Is CUPED Particularly Valuable?

**1. Detecting Small Treatment Effects**

Even experiments with millions of users may struggle to detect very small changes in business metrics. For example, a feature that increases revenue by only 0.1% may require substantial statistical power. CUPED can reduce the Minimum Detectable Effect (MDE) for a fixed sample size.

**2. Experiments With Limited Eligible Populations**

A company may have millions of users overall, but an experiment may target only a small subset.

Examples include:
- Users in a particular geographic market.
- Riders with recent refund requests.
- Users eligible for a specific product flow.

In these cases, the available sample size may be much smaller than the company's total user base. CUPED can help extract more statistical information from the available observations.

**3. High-Variance Metrics**

Metrics such as revenue, gross bookings, and completed trips can vary substantially across users. If historical behavior strongly predicts future outcomes, CUPED may substantially reduce this variation.

**4. Faster Experimentation**

By improving statistical efficiency, CUPED may reduce the sample size or runtime needed to achieve a target level of precision. However, experiments must still satisfy other design requirements, including minimum runtime, seasonality coverage, and appropriate statistical procedures.

## 3.3 When Is CUPED Less Necessary?

CUPED may offer limited incremental value when:

- The experiment already has a very narrow confidence interval.
- The sample size is sufficient for the business decision.
- Historical covariates have weak predictive power.
- The additional implementation complexity is not justified by the precision improvement.

CUPED should not be applied simply because it is statistically sophisticated. Instead, ask:

1. Is our treatment effect estimate already precise enough for the decision?
2. Is the outcome metric highly variable?
3. Do we have strong pre-treatment predictors?
4. Would a narrower confidence interval materially improve our ability to make a decision?


# 4. How to use CUPED in Causal Inference?

## 4.1 Randomized A/B Experiments: The Primary Use Case

CUPED is particularly useful when:

- Treatment assignment is randomized.
- Relevant pre-treatment covariates are available.
- Those covariates strongly predict the outcome.
- The covariates cannot be affected by the experimental treatment.

### Example: Rider Experiment for a Ride-Sharing App

Suppose a ride-sharing App tests whether a new rider feature increases trip frequency.

| Component | Definition |
|---|---|
| Randomization unit | Rider |
| Treatment | Access to the new feature |
| Control | Existing experience |
| Primary metric | Completed trips per rider over 28 days |
| CUPED covariate | Completed trips per rider during the previous 28 days |

Historical trips are likely to predict future trips because rider activity often persists over time. Therefore, CUPED may substantially reduce variance in the primary metric.

### Why Does Randomization Matter?

In a properly randomized experiment, treatment assignment is independent of pre-treatment covariates in expectation:

$$
T \perp X
$$

This means that adjusting for pre-treatment variables does not systematically favor either experimental group. Under appropriate implementation, CUPED can improve precision without introducing systematic bias into the estimated average treatment effect.

Importantly, independence holds **in expectation**, not necessarily in every realized sample. Small baseline differences between Treatment and Control can occur by chance.

### When Is CUPED Less Useful?

CUPED may offer limited benefits when:

**1. Historical data is unavailable**

For example, users who have just signed up may have no prior trip or revenue history. In this case, other pre-treatment characteristics may still be useful for regression adjustment.

**2. Historical data does not predict the outcome**

For example, historical trip frequency may be a weak predictor of whether a user adopts a completely new product category. If the correlation between the covariate and outcome is low, variance reduction will also be limited.

**3. The covariate is affected by treatment**

Variables measured after exposure can be influenced by the experiment itself. Using such variables in a CUPED adjustment can introduce bias.

**4. The metric requires specialized statistical treatment**

For example, ratio metrics such as revenue per order may require adjustments that account for both the numerator and denominator, rather than a naive user-level transformation.

## 4.2 Observational Studies: Not a Standalone Causal Inference Method

CUPED was developed primarily for randomized experiments. In observational studies, treatment is not randomly assigned.

For example
- Suppose a ride-sharing app launches a new ride product like VIP ride, and users independently decide whether to adopt it.
- Users who adopt may already be more active, higher spending, more engaged with the platform, and more likely to increase their spending even without the product.
- If we directly compare adopters with non-adopters, the difference in revenue may reflect both the true product effect and pre-existing differences between those users.
- Applying the standard CUPED adjustment does not automatically solve this problem.
- The reason is that: **Reducing outcome variance is not the same as removing confounding or selection bias.**

Pre-treatment covariates can still be useful for statistical adjustment in observational studies. However, estimating a causal effect requires a valid identification strategy and assumptions that address confounding. CUPED is primarily a variance reduction technique, not a method for establishing causal identification.


# 5. Similar Variance Reduction Methods

**CUPED, multivariate regression adjustment, and CUPAC all aim to improve statistical precision in randomized experiments.** Their main difference is how they use pre-treatment information:

- **CUPED:** Uses a historical metric.
- **Multivariate CUPED:** Uses multiple pre-treatment covariates.
- **CUPAC:** Uses predictions from a statistical or machine learning model.

## 5.1 Rank by Industry Usage Frequency

These are the methods most relevant to compare with CUPED. They belong to the same broad family of **covariate-based variance reduction techniques**. They differ primarily in **how they construct and use predictive covariates**.

The following ranking is a **qualitative assessment of practical adoption and implementation complexity**, based on publicly documented experimentation practices. It is not an industry-wide measured ranking.

| Rank | Method | Approximate Industry Adoption | Main Advantage | When to Use |
|---|---|---|---|---|
| **1** | **CUPED / Single-Covariate ANCOVA** | **Very High** | Simple, interpretable, scalable variance reduction | When a strong pre-treatment predictor exists, especially the historical version of the outcome metric |
| **2** | **Multivariate CUPED / Regression Adjustment** | **High** | Combines multiple predictive covariates | When several pre-treatment variables jointly explain more outcome variation than one variable |
| **3** | **CUPAC (ML-Based Covariate Adjustment)** | **More Specialized** | Captures nonlinear relationships and complex predictive patterns | When historical metrics are insufficient and an accurate predictive model can improve precision |

**Important:** CUPED and ANCOVA are closely related formulations of covariate adjustment. Multivariate CUPED extends this idea to several covariates, while CUPAC uses model predictions as covariates. These are not three completely independent statistical frameworks.


## 5.2 CUPED / ANCOVA: The Standard Starting Point

### What Is It?

Traditional CUPED uses a pre-experiment covariate to adjust the post-experiment outcome. ANCOVA (Analysis of Covariance) provides a closely related regression-based formulation.

A simple ANCOVA model is:

$$
Y_i = \alpha + \tau T_i + \beta X_i + \epsilon_i
$$

where:

- $Y_i$ = Post-experiment outcome.
- $T_i$ = Randomized treatment assignment.
- $X_i$ = Pre-treatment covariate.
- $\tau$ = Estimated treatment effect.

Both methods use the predictive relationship between $X$ and $Y$ to improve treatment effect estimation. Under common implementations, they produce very similar results.

### When Should We Use It?

Use CUPED or simple ANCOVA when:

- The experiment is properly randomized.
- A strong historical predictor exists.
- The relationship between the covariate and outcome is adequately captured by a simple linear adjustment.
- Interpretability and operational simplicity are priorities.

### Example

**Experiment:** Does a new rider incentive increase completed trips?

**Outcome:** Trips completed during the next 28 days.

**Covariate:** Trips completed during the previous 28 days.

Historical trip frequency is a natural starting point because it directly measures the same behavior before treatment.

### Why Is It Popular?

- Relatively easy to implement.
- Computationally efficient.
- Straightforward to explain.
- Can be incorporated into automated experimentation platforms.
- Often provides meaningful precision gains when historical behavior is predictive.

Start with CUPED or ANCOVA when you have a strong historical version of your primary metric.


## 5.3 Multivariate CUPED / Regression Adjustment

### What Is It?

Traditional CUPED often uses one historical metric. However, multiple pre-treatment variables may help predict the outcome. Multivariate regression adjustment extends CUPED by incorporating several covariates.

The adjusted outcome can be expressed as:

$$
Y_{\text{adjusted}}
=
Y
-
\boldsymbol{\theta}^{T}
(\mathbf{X}-E[\mathbf{X}])
$$

where:

- $\mathbf{X}$ is a vector of pre-treatment covariates.
- $\boldsymbol{\theta}$ is a vector of adjustment coefficients.

Instead of relying on one historical metric, the adjustment uses several variables jointly.

### Example

Suppose a rider-sharing app runs an experiment measuring next-28-day rider revenue. Potential pre-treatment covariates include:

- Revenue during the previous 28 days.
- Trips completed during the previous 28 days.
- Rider tenure.
- Historical promotional spending.
- Market characteristics.

Historical revenue may already explain a large portion of future revenue variation. However, the additional variables may explain further variation that historical revenue alone does not capture.

### When Should We Use It Instead of Standard CUPED?

Consider multivariate adjustment when:

- Several covariates are predictive of the outcome.
- Historical outcome data alone does not explain enough variation.
- Additional covariates provide meaningful incremental predictive value.
- The experimentation infrastructure supports reliable multivariate adjustment.

### Trade-Offs

**Advantages:**

- Can explain more outcome variance than a single covariate.
- Can improve precision when multiple variables provide complementary information.
- Remains relatively interpretable compared with complex ML approaches.

**Limitations:**

- More complex than single-covariate CUPED.
- Redundant covariates may provide little additional benefit.
- Unnecessary covariates and overfitting can reduce finite-sample efficiency.

Use multivariate CUPED when additional pre-treatment covariates improve prediction beyond what a single historical metric can achieve.


## 5.4 CUPAC: Machine Learning-Based Variance Reduction

### What Is CUPAC?

**CUPAC (Control Using Predictions as Covariates)** is an extension of CUPED introduced by DoorDash.

Instead of directly using a historical metric as the adjustment covariate, CUPAC uses predictions generated by a machine learning model.

For example, a model might predict each user's expected future revenue using information available before treatment. The prediction becomes the covariate used for variance reduction.

Conceptually:

$$
\hat{Y}_i = f(X_{i1}, X_{i2}, \ldots, X_{ik})
$$

Then:

$$
Y_{\text{adjusted},i}
=
Y_i
-
\theta(\hat{Y}_i-E[\hat{Y}])
$$

where $\hat{Y}_i$ is the model's prediction of the experimental outcome based on valid treatment-unaffected inputs.

### Example

Suppose a food delivery platform runs an experiment to improve delivery completion time.

Delivery time depends on complex factors such as:

- Merchant characteristics.
- Delivery distance.
- Historical traffic patterns.
- Market conditions.
- Time of day.

A single historical metric may not capture these relationships effectively Instead, we could build a predictive model using suitable features that are not affected by the experimental treatment. The model's predicted delivery time could then be used as a covariate to reduce outcome variance.

### When Should We Use CUPAC Instead of CUPED?

Consider CUPAC when:

- The outcome depends on complex or nonlinear relationships.
- No single historical metric is sufficiently predictive.
- A reliable predictive model is available.
- The additional precision benefit justifies model development and maintenance.

### Trade-Offs

**Advantages:**

- Can capture nonlinear relationships.
- Can combine many predictive features into one adjustment covariate.
- Can work well for operational metrics without a natural historical equivalent.

**Limitations:**

- Requires model training, validation, and maintenance.
- More complex to implement and debug.
- Can introduce problems if features are affected by treatment.
- Requires care to avoid overfitting or data leakage.

Model predictions should be evaluated using appropriate out-of-sample validation or sample-splitting procedures.

Use CUPAC when a predictive model offers meaningful incremental variance reduction beyond simpler covariate adjustments.


# 6. Practical Decision Guide

For a randomized experiment:

| Situation | Recommended Starting Point |
|---|---|
| Strong historical version of the primary metric exists | **CUPED / ANCOVA** |
| Several pre-treatment variables provide additional predictive power | **Multivariate CUPED / Regression Adjustment** |
| Outcome has complex nonlinear predictors | **CUPAC** |
| Pre-treatment covariates have little predictive power | **Unadjusted experiment analysis may be sufficient** |
| No historical outcome data exists, but other predictive baseline features are available | **Regression Adjustment or CUPAC** |

### Sample Workflows

1. **Start with the randomized experiment design.**
2. **Identify valid pre-treatment covariates.**
3. **Evaluate how well those covariates predict the primary outcome.**
4. **Start with simple CUPED or ANCOVA when appropriate.**
5. **Consider multivariate adjustment if additional covariates improve precision.**
6. **Consider CUPAC when nonlinear prediction provides meaningful additional value.**
7. **Compare adjusted and unadjusted estimates and validate the standard errors.**

More complex methods are not automatically better. In practice, a good default is to begin with simple, interpretable adjustment and introduce additional complexity only when it delivers meaningful improvements in precision. The goal is to estimate treatment effects as precisely and reliably as possible.

---

## References

1. [Microsoft: Improving the Sensitivity of Online Controlled Experiments by Utilizing Pre-Experiment Data (2013)](https://www.exp-platform.com/Documents/2013-02-CUPED-ImprovingSensitivityOfControlledExperiments.pdf)
2. [DoorDash: Improving Experimental Power through CUPAC](https://careersatdoordash.com/blog/improving-experimental-power-through-control-using-predictions-as-covariate-cupac/)
3. [DoorDash: Meet Dash-AB — The Statistics Engine of Experimentation](https://careersatdoordash.com/blog/meet-dash-ab-the-statistics-engine-of-experimentation-at-doordash/)
4. [Statsig: Variance Reduction](https://docs.statsig.com/experiments/statistical-methods/variance-reduction)

# Section 2. CUPED: Simulation and Implementation

## 2.1 Objective

Compare treatment effect estimates, standard errors, and confidence intervals before and after applying CUPED.

## 2.2 Simulation Setup

We simulate a randomized A/B experiment with 20,000 users to demonstrate how CUPED reduces variance. The experiment has the following setup:

- **Randomization unit:** User
- **Treatment assignment:** 50% Treatment / 50% Control
- **Pre-experiment covariate:** Revenue in the previous 28 days
- **Outcome metric:** Revenue in the following 28 days
- **True treatment effect:** +$5 per user

We generate post-experiment revenue using:

$$
Y = 0.8X + 5T + \epsilon
$$

where:

- $Y$ = Post-experiment revenue
- $X$ = Pre-experiment revenue
- $T$ = Randomized treatment assignment
- $\epsilon$ = Random noise
- $5$ = True treatment effect

We intentionally make historical revenue strongly correlated with future revenue. This reflects a realistic scenario: users who spent more in the past often tend to spend more in the future. Because historical revenue explains a substantial portion of outcome variation, CUPED should be able to remove much of this predictable variation.

In [3]:
import pandas as pd
import numpy as np

In [4]:
np.random.seed(42)
n = 20000

pre_revenue = np.random.lognormal(mean=4, sigma=0.8, size=n)
treatment = np.random.binomial(1, 0.5, n)
noise = np.random.normal(0, 30, n)

post_revenue = 0.8*pre_revenue + 5*treatment + noise
df = pd.DataFrame({"treatment": treatment, "pre_28_revenue": pre_revenue, "post_28_revenue": post_revenue})
df.head()

,treatment,pre_28_revenue,post_28_revenue
0,0,81.237042,28.613210
1,1,48.880990,37.810441
2,1,91.665935,117.472447
3,1,184.642936,167.917421
4,1,45.271482,4.042328


In [5]:
X = df["pre_28_revenue"]
Y = df["post_28_revenue"]

theta = np.cov(X, Y, ddof=1)[0,1]/np.var(X, ddof=1)
df["post_28_revenue_cuped"] = Y - theta * (X - X.mean())
df.head()

,treatment,pre_28_revenue,post_28_revenue,post_28_revenue_cuped
0,0,81.237042,28.613210,24.171238
1,1,48.880990,37.810441,59.246144
2,1,91.665935,117.472447,104.689670
3,1,184.642936,167.917421,80.773636
4,1,45.271482,4.042328,28.364837


In [6]:
raw_effect = df.loc[df["treatment"]==1, "post_28_revenue"].mean() - df.loc[df["treatment"]==0, "post_28_revenue"].mean()
cuped_effect = df.loc[df["treatment"]==1, "post_28_revenue_cuped"].mean() - df.loc[df["treatment"]==0, "post_28_revenue_cuped"].mean()
raw_var = df["post_28_revenue"].var()
cuped_var = df["post_28_revenue_cuped"].var()
variance_reduction = 1 - cuped_var / raw_var
correlation = df[["pre_28_revenue", "post_28_revenue"]].corr()

In [7]:
print("Raw effect:", raw_effect)
print("CUPED effect:", cuped_effect)
print("Raw variance:", raw_var)
print("CUPED variance:", cuped_var)
print("Variance reduction:", variance_reduction)
print("Pre vs. Post revenue correlation:", correlation)

Raw effect: 4.454206360889486
CUPED effect: 4.756700319700094
Raw variance: 4295.038316053478
CUPED variance: 902.7834320848782
Variance reduction: 0.7898078280907151
Pre vs. Post revenue correlation:                  pre_28_revenue  post_28_revenue
pre_28_revenue         1.000000         0.888711
post_28_revenue        0.888711         1.000000


**Insights**
- If a user had unusually high pre-period revenue, CUPED expects part of their high post-period revenue to come from that baseline behavior and subtracts that predictable component.
- If a user had unusually low pre-period revenue, it adjusts upward.
- So CUPED is essentially removing baseline-driven heterogeneity from the outcome. It does not try to change the treatment effect; it tries to reduce noise so the Treatment vs. Control comparison becomes more precise.
- CUPED removes explainable noise, reduces variance and standard error, and therefore increases statistical power.

In [8]:
def calculate_ci(df, outcome_col):
    treated = df.loc[df["treatment"] == 1, outcome_col]
    control = df.loc[df["treatment"] == 0, outcome_col]

    effect = treated.mean() - control.mean()

    se = np.sqrt(
        treated.var(ddof=1) / len(treated)
        + control.var(ddof=1) / len(control)
    )

    ci_lower = effect - 1.96 * se
    ci_upper = effect + 1.96 * se

    return effect, se, (ci_lower, ci_upper)


raw_effect, raw_se, raw_ci = calculate_ci(df, "post_28_revenue")

cuped_effect, cuped_se, cuped_ci = calculate_ci(
    df, "post_28_revenue_cuped"
)

print(f"Raw estimate:   {raw_effect:.3f}")
print(f"Raw SE:         {raw_se:.3f}")
print(f"Raw 95% CI:     {raw_ci}")

print(f"\nCUPED estimate: {cuped_effect:.3f}")
print(f"CUPED SE:       {cuped_se:.3f}")
print(f"CUPED 95% CI:   {cuped_ci}")

Raw estimate:   4.454
Raw SE:         0.926
Raw 95% CI:     (np.float64(2.638676171941257), np.float64(6.269736549837716))

CUPED estimate: 4.757
CUPED SE:       0.424
CUPED 95% CI:   (np.float64(3.9263569793978776), np.float64(5.587043660002311))


## 2.3 Results and Interpretation

### Treatment Effect Estimation

The true treatment effect in our simulation is **+$5 per user**.

Our estimated treatment effects are:

- **Without CUPED:** +$4.45

- **With CUPED:** +$4.76

Both estimates are reasonably close to the true treatment effect. Although the CUPED estimate happens to be closer to $5 in this simulation, this is not the primary purpose of CUPED. **CUPED does not guarantee that the estimated treatment effect will be closer to the true effect in every experiment.** Its main purpose is to reduce the variance of the estimator.

### Variance Reduction

Our results show:

- **Original outcome variance:** 4,295.04
- **CUPED-adjusted outcome variance:** 902.78
- **Variance reduction:** 78.98%

This substantial reduction occurs because pre-experiment revenue strongly predicts post-experiment revenue. The correlation between historical and future revenue is approximately:

$$
\rho = 0.889
$$

The theoretical variance reduction is:

$$
\rho^2 = 0.889^2 \approx 79\%
$$

This closely matches our observed variance reduction.

### Confidence Interval Comparison

By reducing outcome variance, CUPED also reduces the standard error of the treatment effect estimate. As a result, the 95% confidence interval becomes substantially narrower. This allows us to estimate the treatment effect more precisely **without increasing the sample size**.

### Key Takeaway

CUPED improves statistical precision by removing predictable variation in the outcome using pre-treatment covariates. In this simulation:

1. Randomization identifies the treatment effect.
2. Historical revenue explains much of the variation in future revenue.
3. CUPED removes a substantial portion of that predictable variation.
4. The treatment effect estimate becomes more precise, with a smaller standard error and narrower confidence interval.

**CUPED improves precision, not causal identification.**